In [ ]:
"""Notebook: EDA - Análise Exploratória de Dados de Reviews Médicos.

Exploração do dataset de avaliações médicas com foco em:
1. Distribuição de sentimentos
2. Análise de comprimento de textos
3. Distribuição de classes
4. Exemplos de cada classe
"""

# %% Cell 1: Setup e imports
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Setup path
REPO_ROOT = Path.cwd().parent
sys.path.insert(0, str(REPO_ROOT))

from src.config import settings
from src.data.dataset import prepare_dataset
from src.data.preprocessing import DataPreprocessor

plt.style.use("seaborn-v0_8-darkgrid")
sns.set_palette("husl")

print("Setup completo. Iniciando EDA...")

# %% Cell 2: Download e carregamento de dados
print("\n=== ETAPA 1: CARREGAMENTO DE DADOS ===\n")

dataset_path = prepare_dataset(force=False)
print(f"✓ Dataset carregado de: {dataset_path}")

df = pd.read_csv(dataset_path)
print(f"\nDimensões iniciais: {df.shape}")
print(f"\nColunas: {list(df.columns)}")
print(f"\nPrimeiras amostras:\n{df.head()}")

# %% Cell 3: Limpeza e exploração
print("\n=== ETAPA 2: LIMPEZA E VALIDAÇÃO ===\n")

preprocessor = DataPreprocessor()
df_clean = preprocessor.clean(df)

print(f"Dimensões após limpeza: {df_clean.shape}")
print(f"Linhas removidas: {len(df) - len(df_clean)}")
print(f"Duplicatas removidas: {df['text'].duplicated().sum()}")

# %% Cell 4: Distribuição de sentimentos
print("\n=== ETAPA 3: DISTRIBUIÇÃO DE SENTIMENTOS ===\n")

sentiment_counts = df_clean["sentiment"].value_counts()
sentiment_pct = df_clean["sentiment"].value_counts(normalize=True) * 100

print("Contagem por sentimento:")
for sentiment, count in sentiment_counts.items():
    pct = sentiment_pct[sentiment]
    print(f"  {sentiment:10s}: {count:5d} ({pct:5.1f}%)")

# Visualização
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

sentiment_counts.plot(kind="bar", ax=ax1, color=["#e74c3c", "#f39c12", "#2ecc71"])
ax1.set_title("Distribuição de Sentimentos (contagem)", fontsize=12, fontweight="bold")
ax1.set_ylabel("Número de amostras")
ax1.set_xlabel("Sentimento")
ax1.tick_params(axis="x", rotation=45)

sentiment_pct.plot(
    kind="pie",
    ax=ax2,
    autopct="%1.1f%%",
    colors=["#e74c3c", "#f39c12", "#2ecc71"],
)
ax2.set_title("Distribuição de Sentimentos (%)", fontsize=12, fontweight="bold")
ax2.set_ylabel("")

plt.tight_layout()
plt.savefig(settings.models_dir / "01_sentiment_distribution.png", dpi=100)
print("\n✓ Gráfico salvo: 01_sentiment_distribution.png")

# %% Cell 5: Análise de comprimento de textos
print("\n=== ETAPA 4: ANÁLISE DE COMPRIMENTO DE TEXTOS ===\n")

df_clean["text_length"] = df_clean["text"].str.len()
df_clean["word_count"] = df_clean["text"].str.split().str.len()

print("Estatísticas de comprimento (caracteres):")
print(df_clean["text_length"].describe())

print("\nEstatísticas de comprimento (palavras):")
print(df_clean["word_count"].describe())

# Visualização
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# Histograma de comprimento por caracteres
df_clean["text_length"].hist(bins=50, ax=axes[0, 0], edgecolor="black")
axes[0, 0].set_title("Distribuição de comprimento (caracteres)")
axes[0, 0].set_xlabel("Número de caracteres")

# Histograma de comprimento por palavras
df_clean["word_count"].hist(bins=50, ax=axes[0, 1], edgecolor="black")
axes[0, 1].set_title("Distribuição de comprimento (palavras)")
axes[0, 1].set_xlabel("Número de palavras")

# Box plot por sentimento (caracteres)
df_clean.boxplot(column="text_length", by="sentiment", ax=axes[1, 0])
axes[1, 0].set_title("Comprimento por sentimento (caracteres)")
axes[1, 0].set_xlabel("Sentimento")

# Box plot por sentimento (palavras)
df_clean.boxplot(column="word_count", by="sentiment", ax=axes[1, 1])
axes[1, 1].set_title("Comprimento por sentimento (palavras)")
axes[1, 1].set_xlabel("Sentimento")

plt.suptitle("")
plt.tight_layout()
plt.savefig(settings.models_dir / "02_text_length_analysis.png", dpi=100)
print("✓ Gráfico salvo: 02_text_length_analysis.png")

# %% Cell 6: Exemplos de cada classe
print("\n=== ETAPA 5: EXEMPLOS POR CLASSE ===\n")

for sentiment in ["positive", "neutral", "negative"]:
    sample = df_clean[df_clean["sentiment"] == sentiment]["text"].iloc[0]
    print(f"\n{sentiment.upper()}:")
    print(f"  {sample[:100]}...")

# %% Cell 7: Split treino/validação/teste
print("\n=== ETAPA 6: SPLIT TREINO/VALIDAÇÃO/TESTE ===\n")

train_df, val_df, test_df = preprocessor.split_data(df_clean)

print(f"Treino:     {len(train_df)} amostras ({100*len(train_df)/len(df_clean):.1f}%)")
print(f"Validação:  {len(val_df)} amostras ({100*len(val_df)/len(df_clean):.1f}%)")
print(f"Teste:      {len(test_df)} amostras ({100*len(test_df)/len(df_clean):.1f}%)")

# Distribuição de sentimentos por split
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for idx, (df_split, title) in enumerate(
    [
        (train_df, "Treino"),
        (val_df, "Validação"),
        (test_df, "Teste"),
    ]
):
    split_counts = df_split["sentiment"].value_counts()
    split_counts.plot(kind="bar", ax=axes[idx], color=["#e74c3c", "#f39c12", "#2ecc71"])
    axes[idx].set_title(f"{title} (n={len(df_split)})")
    axes[idx].set_ylabel("Contagem")
    axes[idx].set_xlabel("Sentimento")
    axes[idx].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.savefig(settings.models_dir / "03_train_val_test_split.png", dpi=100)
print("✓ Gráfico salvo: 03_train_val_test_split.png")

# %% Cell 8: Resumo
print("\n=== RESUMO DA EDA ===\n")
print(f"✓ Dataset carregado: {len(df_clean)} amostras")
print(f"✓ Sentimentos: {df_clean['sentiment'].nunique()} classes")
print(f"✓ Comprimento médio: {df_clean['text_length'].mean():.0f} caracteres")
print(f"✓ Split definido: train={len(train_df)}, val={len(val_df)}, test={len(test_df)}")
print(f"✓ Gráficos salvos em: {settings.models_dir}")
print("\nEDA CONCLUÍDA! ✓")